# 03 – So sánh các thuật toán học máy và chọn mô hình tốt nhất

Notebook này chạy **6 thuật toán học máy** trên cùng một bộ đặc trưng và cùng một cách chia dữ liệu, để so sánh công bằng:

| Tên | Thuật toán | Ý tưởng chính |
|---|---|---|
| E1 | Logistic Regression | Đường phân tách tuyến tính + hàm sigmoid |
| E2 | K-Nearest Neighbors (k=5) | Gán nhãn theo đa số của 5 ảnh gần nhất |
| E3 | Gaussian Naive Bayes | Định lý Bayes, giả định các đặc trưng độc lập |
| E4 | Decision Tree (depth 10) | Chuỗi câu hỏi if/else trên đặc trưng |
| E5 | Random Forest (200 cây) | Nhiều cây quyết định bỏ phiếu (ensemble, bagging) |
| E6 | SVM kernel RBF (C=10) | Siêu phẳng lề lớn nhất trong không gian kernel |

**Quy tắc chọn:** `val_acc` cao nhất (hòa thì `val_loss` thấp hơn), chỉ dùng tập validation. Tập test chỉ dùng một lần cho mô hình thắng.

## 0. Trước khi đọc kết quả: dữ liệu, đặc trưng và các chỉ số

**Dữ liệu: chỉ dùng ảnh gốc, chia theo ảnh gốc.** Bộ dữ liệu Kaggle đã được tạo thêm nhiều bản từ mỗi ảnh gốc (xoay, dịch, lật, thêm nhiễu), và các bản của cùng một ảnh nằm lẫn ở cả train lẫn test. Nếu giữ nguyên, model chỉ cần nhận ra ảnh quen là đạt gần 99%, con số đó không phản ánh khả năng nhận ra ảnh mới. Vì vậy dự án chỉ giữ ảnh gốc (khoảng 1.500 ảnh), gộp train và test rồi chia lại 70% / 15% / 15% theo ảnh gốc, chia đều theo từng loại trái cây. Accuracy thấp hơn trước là **bình thường và đáng tin hơn**.

**Đặc trưng: 50 số tính thủ công từ ảnh** (xem `src/features.py`): histogram màu HSV, mean và std của màu, tỉ lệ điểm ảnh tối và nâu, texture. Khi tách được nền (dựa vào màu viền ảnh), các đặc trưng chỉ tính trên phần quả; trục hue được dịch để màu đỏ không bị cắt đôi ở hai đầu thang.

**Mọi thuật toán nằm trong cùng một Pipeline:** `StandardScaler` rồi bộ phân loại. Chuẩn hoá đưa mỗi đặc trưng về mean 0, std 1, cần thiết cho KNN, SVM và Logistic Regression vì chúng nhạy với thang đo (cây quyết định thì không cần nhưng cũng không bị hại).

**Ba chỉ số trong bảng kết quả:**

| Chỉ số | Ý nghĩa | Cách đọc |
|---|---|---|
| `val_acc` | Tỉ lệ ảnh đoán đúng trên tập validation | Càng cao càng tốt |
| `val_f1` | F1 trung bình của hai lớp (macro) | Gần với accuracy khi hai nhãn cân bằng; thấp hơn hẳn thì model đang bỏ sót một lớp |
| `val_loss` | Log-loss: phạt theo xác suất model gán cho nhãn đúng | **Càng thấp càng tốt.** Đoán sai mà rất tự tin bị phạt nặng, đoán đúng nhưng chỉ 55% cũng bị phạt |

`val_loss` đo chất lượng của **độ tin cậy** (`confidence`) mà API trả về. Hai mô hình cùng accuracy nhưng log-loss khác nhau nghĩa là một mô hình biết mình chắc hay không chắc tốt hơn.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import json
import matplotlib.pyplot as plt
from IPython.display import Image

from src.data_loader import BEST_CONFIG_PATH, REPORTS_DIR
from src.experiments import run_experiments

## 1. Chạy các thí nghiệm

Mô hình thắng sẽ được đánh giá trên tập test và lưu vào `models/best_model.joblib` (API sẽ dùng file này). SVM và KNN có thể chậm hơn các thuật toán còn lại.

In [ ]:
results, info = run_experiments()
results[['name', 'val_acc', 'val_f1', 'val_loss', 'seconds']]

### Cách đọc bảng kết quả

- Bảng đã được sắp xếp: hàng đầu là mô hình thắng theo quy tắc chọn.
- Nhìn **cả ba cột**, không chỉ `val_acc`. Hai mô hình có accuracy gần nhau mà `val_loss` chênh nhiều thì mô hình có log-loss thấp hơn cho xác suất đáng tin hơn.
- Cột `seconds` là thời gian huấn luyện. Khi hai mô hình ngang nhau về chất lượng thì mô hình nhanh và đơn giản hơn là lựa chọn tốt hơn.
- **Chênh lệch nhỏ chưa có ý nghĩa.** Tập validation chỉ có vài trăm ảnh nên accuracy dao động khoảng ±2 điểm phần trăm chỉ vì ngẫu nhiên. Nếu hai mô hình hơn kém nhau dưới 2 điểm thì chưa thể nói mô hình nào thật sự tốt hơn.

Ô dưới đây tóm tắt nhanh bảng kết quả vừa chạy.

In [ ]:
best, worst = results.iloc[0], results.iloc[-1]
print(f"Thắng   : {best['name']}  val_acc={best['val_acc']:.2%}  val_loss={best['val_loss']:.3f}")
if len(results) > 1:
    second = results.iloc[1]
    gap = (best['val_acc'] - second['val_acc']) * 100
    print(f"Hạng 2  : {second['name']}  val_acc={second['val_acc']:.2%}  val_loss={second['val_loss']:.3f}")
    print(f"Kém nhất: {worst['name']}  val_acc={worst['val_acc']:.2%}  val_loss={worst['val_loss']:.3f}")
    print(f"Chênh lệch giữa hạng 1 và hạng 2: {gap:.2f} điểm %")
    if gap < 2:
        print('-> Dưới 2 điểm %: hai mô hình gần như ngang nhau, nên xét thêm val_loss và tốc độ.')
    else:
        print('-> Mô hình thắng hơn rõ rệt.')

In [ ]:
Image(os.path.join(REPORTS_DIR, 'experiments_comparison.png'))

Biểu đồ cột ngang: mỗi thanh là `val_acc` của một thí nghiệm, kèm log-loss trong ngoặc. Trục bắt đầu từ một mức cao hơn 0 nên các thanh nhìn có vẻ chênh nhiều hơn thực tế, hãy đọc số ghi ở đầu thanh.

### 1.1 Giải thích từng thí nghiệm

Mỗi mục dưới đây trả lời bốn câu hỏi: thuật toán hoạt động thế nào, siêu tham số có nghĩa gì, hợp hay không hợp với dữ liệu của dự án, và nên chú ý gì khi đọc kết quả.

#### E1 – Logistic Regression (C = 1.0): mô hình baseline

- **Cách hoạt động:** cộng các đặc trưng (đã chuẩn hoá) có nhân trọng số, rồi đưa qua hàm sigmoid để ra xác suất `rotten`. Ranh giới giữa `fresh` và `rotten` là một mặt phẳng thẳng trong không gian 50 chiều.
- **Siêu tham số `C`:** là nghịch đảo của độ mạnh regularization. `C` nhỏ thì ép các hệ số nhỏ lại, mô hình đơn giản hơn; `C` lớn thì cho phép khớp dữ liệu train sát hơn.
- **Hợp với dự án ở đâu:** nhanh, ổn định, xác suất tương đối hợp lý, và các hệ số đọc được (xem notebook 02).
- **Hạn chế:** ranh giới chỉ là đường thẳng, trong khi màu của quả tươi và quả hỏng chồng lấn nhau theo kiểu phức tạp (ví dụ táo đỏ thẫm còn tươi và táo đỏ đã bắt đầu thâm).
- **Vai trò trong thí nghiệm:** làm mốc so sánh. Một thuật toán phức tạp hơn mà không thắng nổi E1 thì không đáng dùng.

#### E2 – K-Nearest Neighbors (k = 5)

- **Cách hoạt động:** không có bước học. Khi gặp ảnh mới, thuật toán tìm 5 ảnh train có vector đặc trưng gần nhất (khoảng cách Euclid) và lấy nhãn chiếm đa số.
- **Vì sao cần `StandardScaler`:** khoảng cách bị đặc trưng có thang đo lớn chi phối, nên phải chuẩn hoá trước.
- **Xác suất rất thô:** xác suất chính là tỉ lệ láng giềng cùng nhãn, nên với k = 5 chỉ có thể là 0%, 20%, 40%, 60%, 80% hoặc 100%. Vì vậy log-loss của KNN thường cao hơn SVM hay Random Forest dù accuracy gần nhau.
- **Hạn chế:** nhạy với các đặc trưng ít thông tin (nhiều bin histogram gần như luôn bằng 0), và dự đoán chậm khi dữ liệu lớn.
- **Chọn `k`:** xem mục 3.

#### E3 – Gaussian Naive Bayes

- **Cách hoạt động:** với mỗi lớp, mô hình hoá từng đặc trưng bằng một phân phối chuẩn riêng, rồi nhân các xác suất lại với nhau theo định lý Bayes. Điều kiện để cách làm này đúng là **các đặc trưng độc lập với nhau**.
- **Vì sao không hợp với dự án:** giả định độc lập bị vi phạm rõ. Các bin trong cùng một histogram có tổng bằng 1 nên phụ thuộc nhau; mean của R, G, B và của H, S, V liên quan chặt chẽ; và nhiều bin histogram không có dạng phân phối chuẩn (rất nhiều giá trị bằng 0).
- **Hệ quả thường thấy:** accuracy thấp nhất nhóm, và xác suất quá tự tin (log-loss lớn) vì thông tin trùng lặp bị tính nhiều lần.
- **Vai trò:** giữ lại như một đại diện cho nhóm mô hình xác suất đơn giản, và để thấy giả định sai làm hỏng kết quả ra sao.

#### E4 – Decision Tree (max_depth = 10)

- **Cách hoạt động:** học một chuỗi câu hỏi dạng *tỉ lệ điểm ảnh nâu có lớn hơn một ngưỡng không?*, chia dữ liệu dần cho đến khi mỗi nhánh gần như chỉ còn một nhãn.
- **Siêu tham số `max_depth`:** giới hạn số câu hỏi liên tiếp. Không giới hạn thì cây học thuộc từng ảnh train (overfitting). Xem thực nghiệm ở mục 4.
- **Điểm mạnh:** dễ giải thích, không cần chuẩn hoá đặc trưng.
- **Điểm yếu:** một cây rất không ổn định (đổi vài ảnh train có thể ra cây khác hẳn). Xác suất ở mỗi lá thường gần 0 hoặc gần 1, nên khi đoán sai thì sai rất tự tin và **`val_loss` thường cao** dù accuracy không tệ.

#### E5 – Random Forest (200 cây)

- **Cách hoạt động:** huấn luyện 200 cây, mỗi cây học trên một mẫu ngẫu nhiên (có lặp) của tập train và ở mỗi lần chia chỉ được xét một phần ngẫu nhiên của đặc trưng. Kết quả cuối là tỉ lệ cây bỏ phiếu cho từng lớp.
- **Vì sao tốt hơn một cây:** lấy trung bình nhiều cây khác nhau làm giảm độ dao động (variance) của từng cây, nên ít overfit hơn nhưng vẫn bắt được quan hệ phi tuyến giữa các đặc trưng.
- **Cách đọc xác suất:** với Random Forest, xác suất chính là tỉ lệ cây bỏ phiếu. Kết quả 51% `rotten` và 49% `fresh` nghĩa là gần một nửa số cây chọn mỗi bên, tức ảnh đó mơ hồ hoặc khác kiểu dữ liệu đã học.
- **Siêu tham số `n_estimators`:** nhiều cây hơn thì ổn định hơn nhưng lợi ích giảm dần và huấn luyện lâu hơn.
- **Bonus:** cho biết đặc trưng nào quan trọng (mục 5).

#### E6 – SVM kernel RBF (C = 10)

- **Cách hoạt động:** tìm ranh giới tách hai lớp với **lề (khoảng cách tới ảnh gần nhất) lớn nhất**. Kernel RBF cho phép ranh giới cong, bằng cách đo độ giống nhau giữa hai ảnh theo khoảng cách trong không gian đặc trưng.
- **Siêu tham số `C`:** mức phạt cho ảnh nằm sai phía ranh giới. `C` lớn (như 10) thì ranh giới uốn theo từng ảnh train, khớp chặt hơn nhưng có nguy cơ overfit; `C` nhỏ thì lề rộng và đơn giản hơn.
- **Siêu tham số `gamma='scale'`:** độ rộng vùng ảnh hưởng của mỗi ảnh, được tự tính từ dữ liệu.
- **Về xác suất:** SVM bản thân chỉ cho biết ảnh nằm phía nào của ranh giới. Với `probability=True`, scikit-learn thêm một bước hiệu chỉnh (Platt scaling, dùng cross-validation bên trong) để đổi khoảng cách tới ranh giới thành xác suất, nên huấn luyện chậm hơn và ảnh nằm sát ranh giới sẽ có xác suất gần 50%.
- **Điểm mạnh:** thường rất tốt với vài nghìn mẫu và vài chục đặc trưng như bài này. **Điểm yếu:** chậm khi dữ liệu lớn, khó giải thích.

## 2. Kết quả chốt

Cấu hình thắng, điểm validation và điểm test cuối cùng.

In [ ]:
print(json.dumps(info, ensure_ascii=False, indent=2))
Image(os.path.join(REPORTS_DIR, 'confusion_matrix.png'))

### Cách đọc kết quả chốt

**Validation và test khác nhau thế nào?** Mô hình thắng được *chọn* dựa trên điểm validation, nên điểm đó hơi lạc quan (mô hình thắng một phần nhờ may mắn trên tập này). Tập test chưa từng được dùng để chọn nên điểm test là con số trung thực hơn. Test thấp hơn validation vài điểm phần trăm là chuyện bình thường. Test chỉ được dùng **một lần**; nếu chỉnh mô hình lại sau khi xem điểm test thì test không còn khách quan nữa.

**Confusion matrix (ma trận nhầm lẫn):** hàng là nhãn thật, cột là nhãn dự đoán. Ô trên đường chéo là dự đoán đúng. Hai ô còn lại là hai loại lỗi khác nhau:

| Lỗi | Nghĩa | Hậu quả |
|---|---|---|
| `fresh` bị đoán là `rotten` | Báo nhầm quả tốt là hỏng | Bỏ phí quả còn dùng được |
| `rotten` bị đoán là `fresh` | Bỏ sót quả hỏng | Nguy hiểm hơn nếu dùng để kiểm tra an toàn thực phẩm |

Vì vậy ngoài accuracy, nên nhìn **recall của lớp `rotten`** trong `reports/classification_report.txt`: trong tất cả quả thật sự hỏng, model bắt được bao nhiêu phần trăm.

**Độ chắc chắn của con số:** tập test chỉ có vài trăm ảnh, nên accuracy có sai số khoảng ±2 điểm phần trăm (1 độ lệch chuẩn), khoảng tin cậy 95% xấp xỉ ±4 điểm. Khi viết báo cáo nên ghi kèm số ảnh test và không khẳng định hơn kém khi chênh lệch nhỏ hơn mức này.

**Phạm vi áp dụng:** điểm số này đo trên ảnh **cùng kiểu** với dữ liệu huấn luyện (quả nguyên, nền sáng, ba loại: táo, chuối, cam). Ảnh khác kiểu hoặc loại trái cây chưa có trong dữ liệu có thể cho kết quả kém hơn nhiều, thường thể hiện qua xác suất gần 50/50.

## 3. Chọn siêu tham số: ảnh hưởng của `k` trong KNN

**Bias–variance:** `k` nhỏ thì mô hình rất linh hoạt (dễ overfit, accuracy train cao nhưng val thấp); `k` lớn thì mô hình mượt hơn nhưng có thể quá đơn giản (underfit). Điểm tốt nhất thường nằm ở giữa.

In [ ]:
from sklearn.metrics import accuracy_score
from src.features import load_feature_splits
from src.train import build_model

data = load_feature_splits()
X_train, y_train = data['train']
X_val, y_val = data['val']

ks = [1, 3, 5, 7, 11, 15, 21, 31]
tr_acc, va_acc = [], []
for k in ks:
    m = build_model({'algo': 'knn', 'params': {'n_neighbors': k}}).fit(X_train, y_train)
    tr_acc.append(accuracy_score(y_train, m.predict(X_train)))
    va_acc.append(accuracy_score(y_val, m.predict(X_val)))

plt.plot(ks, tr_acc, marker='o', label='Train')
plt.plot(ks, va_acc, marker='o', label='Validation')
plt.xlabel('k (số láng giềng)'); plt.ylabel('Accuracy'); plt.title('KNN: ảnh hưởng của k')
plt.legend(); plt.grid(alpha=0.3); plt.show()

### Cách đọc biểu đồ KNN

- **Chỉ có đường Validation mới cho biết chất lượng thật.** Với `k = 1`, đường Train gần như chắc chắn bằng 100%, vì mỗi ảnh train có chính nó là láng giềng gần nhất. Đó là ảnh học thuộc, không phải khả năng nhận ra ảnh mới.
- **Khoảng cách giữa hai đường** là mức overfit. Khoảng cách lớn ở `k` nhỏ rồi thu hẹp dần khi `k` tăng là hình dạng bình thường.
- **Cách chọn:** lấy `k` cho validation cao nhất. Nếu vài giá trị `k` cho điểm gần như bằng nhau thì chọn `k` lớn hơn, vì mô hình mượt và ổn định hơn. Khi `k` quá lớn, cả hai đường cùng giảm: mô hình đã quá đơn giản (underfit).
- **Đối chiếu với E2:** thí nghiệm E2 dùng `k = 5`. Nếu đỉnh validation ở biểu đồ nằm ở `k` khác hẳn và hơn rõ rệt, nên cập nhật `EXPERIMENTS` trong `src/experiments.py`. Việc chọn `k` chỉ dựa trên validation, **không** dùng test.

## 4. Overfitting với Decision Tree: ảnh hưởng của `max_depth`

Cây càng sâu càng khớp chặt dữ liệu train. Khi khoảng cách giữa đường Train và Validation lớn dần là dấu hiệu **overfitting**.

In [ ]:
depths = [1, 2, 3, 5, 8, 12, 20, None]
tr_acc, va_acc = [], []
for d in depths:
    m = build_model({'algo': 'decision_tree', 'params': {'max_depth': d}}).fit(X_train, y_train)
    tr_acc.append(accuracy_score(y_train, m.predict(X_train)))
    va_acc.append(accuracy_score(y_val, m.predict(X_val)))

labels = [str(d) for d in depths]
plt.plot(labels, tr_acc, marker='o', label='Train')
plt.plot(labels, va_acc, marker='o', label='Validation')
plt.xlabel('max_depth (None = không giới hạn)'); plt.ylabel('Accuracy'); plt.title('Decision Tree: ảnh hưởng của độ sâu')
plt.legend(); plt.grid(alpha=0.3); plt.show()

### Cách đọc biểu đồ Decision Tree

Biểu đồ thường có ba vùng:

| Vùng | Dấu hiệu | Ý nghĩa |
|---|---|---|
| Cây nông (`max_depth` 1–3) | Train và Validation **đều thấp** | **Underfitting:** cây quá đơn giản, chưa đủ câu hỏi để tách hai lớp |
| Cây vừa | Validation đạt đỉnh hoặc bắt đầu đi ngang, khoảng cách với Train còn nhỏ | **Vùng tốt nhất** |
| Cây sâu (`max_depth` lớn, `None`) | Train tiến gần 100% nhưng Validation không tăng thêm hoặc giảm | **Overfitting:** cây học thuộc ảnh train, kể cả nhiễu |

- **Chọn độ sâu:** lấy điểm mà Validation bắt đầu đi ngang. E4 dùng `max_depth = 10` như một mức vừa phải; nếu biểu đồ cho thấy validation đạt đỉnh ở độ sâu khác rõ rệt thì nên cân nhắc đổi.
- **Liên hệ với Random Forest (E5):** thay vì cố tìm một cây có độ sâu vừa vặn, Random Forest dùng nhiều cây sâu rồi lấy trung bình. Việc lấy trung bình triệt tiêu phần lớn overfitting của từng cây, và đó là lý do E5 thường vượt E4.

## 5. Độ quan trọng của đặc trưng

Nếu mô hình thắng là Random Forest hoặc Decision Tree, biểu đồ này cho biết đặc trưng nào đóng góp nhiều nhất vào việc phân biệt `fresh` và `rotten`.

In [ ]:
path = os.path.join(REPORTS_DIR, 'feature_importance.png')
Image(path) if os.path.exists(path) else print('Mô hình thắng không có feature_importances_ (không phải thuật toán dựa trên cây).')

### Cách đọc biểu đồ độ quan trọng

- **Thang đo:** độ quan trọng của tất cả đặc trưng cộng lại bằng 1. Thanh càng dài nghĩa là các cây dùng đặc trưng đó càng nhiều để tách hai lớp.
- **Ý nghĩa tên đặc trưng:**
  - `hist_H_i`: tỉ lệ điểm ảnh có màu (hue) rơi vào khoảng thứ `i`. Trục hue đã được dịch để màu đỏ liền một khối, nên `i` nhỏ ứng với đỏ và cam, ở giữa ứng với vàng, lớn hơn ứng với xanh lá.
  - `hist_S_i`, `hist_V_i`: tỉ lệ điểm ảnh theo độ bão hoà màu và độ sáng.
  - `mean_*`, `std_*`: màu trung bình và mức biến thiên màu của phần quả.
  - `dark_ratio`, `brown_ratio`: tỉ lệ điểm ảnh tối và điểm ảnh màu nâu.
  - `grad_*`: độ gồ ghề của bề mặt (đốm, vết thâm làm gradient lớn hơn).
- **Cách diễn giải cho báo cáo:** nếu nhóm đứng đầu là đặc trưng về màu nâu, độ tối hay độ sáng thì khớp với trực giác (quả hỏng thường nâu, xỉn, tối). Nếu `grad_*` cao thì bề mặt có đốm và vết là dấu hiệu quan trọng.
- **Cẩn thận khi kết luận:**
  - Độ quan trọng cho biết mô hình **dựa vào** đặc trưng nào, không chứng minh đặc trưng đó gây ra việc quả hỏng.
  - Các đặc trưng tương quan với nhau (ví dụ nhiều bin hue kề nhau) chia sẻ độ quan trọng, nên từng thanh riêng lẻ có thể nhìn nhỏ dù cả nhóm quan trọng.
  - Nếu mô hình học dựa vào nền hay bóng đổ thay vì quả thì chúng vẫn có thể hiện ra là quan trọng. Đó là lý do đặc trưng được tính chỉ trên phần quả.

## 6. Kết luận và hạn chế

**Cách chốt mô hình:** chọn theo `val_acc` cao nhất (hòa thì `val_loss` thấp hơn), chỉ dùng validation; mô hình thắng được đánh giá một lần trên test và lưu vào `models/best_model.joblib` cho API. Khi hai mô hình ngang nhau (chênh dưới khoảng 2 điểm phần trăm) thì ưu tiên mô hình có log-loss thấp hơn, nhanh hơn và đơn giản hơn.

**Hạn chế cần nêu trong báo cáo:**

- Dữ liệu chỉ có **ba loại trái cây** (táo, chuối, cam), khoảng 1.500 ảnh gốc, hầu hết là quả nguyên trên nền sáng. Loại trái khác hoặc ảnh chụp thực tế (nền phức tạp, cầm trên tay, ánh sáng yếu) nằm ngoài phạm vi đã học.
- Đặc trưng thủ công chỉ nắm **màu và độ gồ ghề tổng thể**, không hiểu hình dạng quả hay vị trí vết hỏng, nên các vết nhỏ trên quả còn tươi hoặc quả sẫm màu tự nhiên dễ bị nhầm.
- Con số test đo trên vài trăm ảnh nên có sai số vài điểm phần trăm.

**Hướng cải thiện:**

1. Thêm ảnh thật, đa dạng nền và ánh sáng, cho từng loại trái.
2. Phân tích các ảnh bị đoán sai trên tập test để biết lỗi tập trung ở đâu (loại trái nào, kiểu ảnh nào).
3. Thay đặc trưng thủ công bằng đặc trưng trích từ mạng CNN huấn luyện sẵn (ví dụ MobileNet), rồi dùng lại các thuật toán ở notebook này.
4. Trong API, trả về `uncertain` khi độ tin cậy thấp thay vì ép chọn `fresh` hay `rotten`.